# 01 Data Preparation

Zakres notebooka:
1. Kontrola jakosci danych surowych.
2. Zbudowanie targetu `sell/hold/buy` na calym zbiorze.
3. Ocena potrzeby wazonego probkowania (`WeightedRandomSampler`) przed splitem.
4. Chronologiczny split train/val/test i zapis plikow do `data/processed`.


In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")


In [ ]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise RuntimeError("Could not find project root with data/ and notebooks/ directories.")


PROJECT_ROOT = find_project_root()
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "ethusdt_1h.csv"

raw_df = pd.read_csv(RAW_PATH)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_PATH:", RAW_PATH)
print("Raw rows:", len(raw_df))


### 1. Raw data overview


In [ ]:
display(pd.DataFrame({"rows": [len(raw_df)], "columns": [len(raw_df.columns)]}))
display(pd.DataFrame({"column": raw_df.columns}))
display(raw_df.head(5))
display(raw_df.tail(5))


### 2. Data quality checks


In [ ]:
quality_table = pd.DataFrame(
    {
        "missing_values": raw_df.isna().sum(),
        "dtype": raw_df.dtypes.astype(str),
    }
)

duplicate_rows = int(raw_df.duplicated().sum())
duplicate_timestamps = int(raw_df["timestamp"].duplicated().sum())

print("Duplicate rows:", duplicate_rows)
print("Duplicate timestamps:", duplicate_timestamps)
display(quality_table)


### 3. Timestamp checks


In [ ]:
ts = pd.to_datetime(raw_df["timestamp"], utc=True, errors="coerce")

ts_info = pd.DataFrame(
    {
        "min_timestamp": [ts.min()],
        "max_timestamp": [ts.max()],
        "non_null_count": [int(ts.count())],
        "unique_count": [int(ts.nunique())],
        "missing_count": [int(ts.isna().sum())],
        "duplicate_count": [int(ts.duplicated().sum())],
    }
)
display(ts_info)

full_idx = pd.date_range(ts.min(), ts.max(), freq="h", tz="UTC")
missing_hours = full_idx.difference(pd.DatetimeIndex(ts.dropna()))
print("Missing hours in full range:", len(missing_hours))
display(pd.DataFrame({"missing_hours": missing_hours[:20]}))

obs_per_month = ts.dropna().dt.to_period("M").value_counts().sort_index()
plt.figure(figsize=(12, 4))
obs_per_month.plot(kind="bar", color="#4C78A8")
plt.title("Records per month")
plt.xlabel("month")
plt.ylabel("count")
plt.tight_layout()
plt.show()


### 4. Candle integrity checks


In [ ]:
num_df = raw_df.copy()
for col in ["open", "high", "low", "close", "volume"]:
    num_df[col] = pd.to_numeric(num_df[col], errors="coerce")

invalid_high = int((num_df["high"] < num_df[["open", "close"]].max(axis=1)).sum())
invalid_low = int((num_df["low"] > num_df[["open", "close"]].min(axis=1)).sum())
negative_volume = int((num_df["volume"] < 0).sum())

candles_integrity = pd.DataFrame(
    {
        "invalid_high": [invalid_high],
        "invalid_low": [invalid_low],
        "negative_volume": [negative_volume],
    }
)
display(candles_integrity)


### 5. Local helper functions (simple notebook version)


In [ ]:
@dataclass
class DataConfig:
    raw_csv: Path = PROJECT_ROOT / "data" / "raw" / "ethusdt_1h.csv"
    clean_csv: Path = PROJECT_ROOT / "data" / "interim" / "eth_clean.csv"
    features_parquet: Path = PROJECT_ROOT / "data" / "processed" / "features.parquet"
    train_parquet: Path = PROJECT_ROOT / "data" / "processed" / "train.parquet"
    val_parquet: Path = PROJECT_ROOT / "data" / "processed" / "val.parquet"
    test_parquet: Path = PROJECT_ROOT / "data" / "processed" / "test.parquet"


@dataclass
class TargetConfig:
    horizon: int = 6
    threshold: float = 0.0075


@dataclass
class SplitConfig:
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15


def _save_dataframe(df: pd.DataFrame, output_path: Path) -> None:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if output_path.suffix.lower() == ".parquet":
        df.to_parquet(output_path, index=True)
    elif output_path.suffix.lower() == ".csv":
        df.to_csv(output_path, index=True)
    else:
        raise ValueError(f"Unsupported output format: {output_path.suffix}")


def _clean_ohlcv(df: pd.DataFrame) -> tuple[pd.DataFrame, dict[str, int]]:
    work = df.copy()
    stats: dict[str, int] = {"rows_initial": len(work)}

    work["timestamp"] = pd.to_datetime(work["timestamp"], utc=True, errors="coerce")
    work = work.dropna(subset=["timestamp"]).sort_values("timestamp")

    duplicated_ts = int(work["timestamp"].duplicated().sum())
    stats["duplicated_timestamps"] = duplicated_ts
    work = work.drop_duplicates(subset=["timestamp"], keep="first")

    numeric_cols = ["open", "high", "low", "close", "volume"]
    for col in numeric_cols:
        work[col] = pd.to_numeric(work[col], errors="coerce")

    nan_before = int(work[numeric_cols].isna().sum().sum())
    stats["nan_before_drop"] = nan_before
    work = work.dropna(subset=numeric_cols)

    invalid_candles = (
        (work["high"] < work[["open", "close"]].max(axis=1))
        | (work["low"] > work[["open", "close"]].min(axis=1))
        | (work["volume"] < 0)
    )
    stats["invalid_candles"] = int(invalid_candles.sum())
    work = work.loc[~invalid_candles].copy()

    work = work.set_index("timestamp")
    stats["rows_final"] = len(work)
    return work, stats


def _create_basic_features(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()

    work["return_1"] = work["close"].pct_change(1)
    work["return_6"] = work["close"].pct_change(6)
    work["return_24"] = work["close"].pct_change(24)
    work["log_return_1"] = np.log(work["close"]).diff(1)

    work["close_open_pct"] = (work["close"] - work["open"]) / work["open"]
    work["high_low_pct"] = (work["high"] - work["low"]) / work["close"]

    work["rolling_vol_24"] = work["return_1"].rolling(24).std()
    vol_mean_24 = work["volume"].rolling(24).mean()
    vol_std_24 = work["volume"].rolling(24).std()
    work["volume_zscore_24"] = (work["volume"] - vol_mean_24) / vol_std_24

    hour = work.index.hour
    dow = work.index.dayofweek
    work["hour_sin"] = np.sin(2 * np.pi * hour / 24)
    work["hour_cos"] = np.cos(2 * np.pi * hour / 24)
    work["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    work["dow_cos"] = np.cos(2 * np.pi * dow / 7)

    return work


def _make_multiclass_target(
    df: pd.DataFrame,
    horizon: int = 6,
    threshold: float = 0.0075,
    close_col: str = "close",
) -> pd.DataFrame:
    work = df.copy()
    future_return = work[close_col].shift(-horizon) / work[close_col] - 1.0
    work["future_return"] = future_return

    work["target"] = 1
    work.loc[future_return < -threshold, "target"] = 0
    work.loc[future_return > threshold, "target"] = 2

    return work.iloc[:-horizon].copy()


def _chronological_split(
    df: pd.DataFrame,
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    total = train_ratio + val_ratio + test_ratio
    if not np.isclose(total, 1.0):
        raise ValueError(f"Split ratios must sum to 1.0, got {total}")

    n = len(df)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)

    train_df = df.iloc[:n_train].copy()
    val_df = df.iloc[n_train : n_train + n_val].copy()
    test_df = df.iloc[n_train + n_val :].copy()
    return train_df, val_df, test_df


### 6. Target distribution BEFORE split (project requirement)

Ta sekcja odpowiada wymaganiu projektowemu: liczebnosc klas ma byc policzona przed split,
aby ocenic strategie dzielenia danych i potrzebe `WeightedRandomSampler`.


In [ ]:
clean_df, cleaning_stats = _clean_ohlcv(raw_df)
pre_split_df = _create_basic_features(clean_df)
pre_split_df = _make_multiclass_target(pre_split_df, horizon=6, threshold=0.0075)
pre_split_df = pre_split_df.dropna(subset=["future_return", "target"]).copy()

label_map = {0: "sell", 1: "hold", 2: "buy"}
class_counts = pre_split_df["target"].value_counts().sort_index()
class_share = (class_counts / class_counts.sum() * 100).round(2)

class_table = pd.DataFrame(
    {
        "label": [label_map.get(i, str(i)) for i in class_counts.index],
        "count": class_counts.values,
        "share_pct": class_share.values,
    },
    index=class_counts.index,
)
display(class_table)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.barplot(
    x=[label_map.get(i, str(i)) for i in class_counts.index],
    y=class_counts.values,
    hue=[label_map.get(i, str(i)) for i in class_counts.index],
    palette="Blues_d",
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Class distribution before split")
axes[0].set_xlabel("class")
axes[0].set_ylabel("count")

sns.histplot(pre_split_df["future_return"], bins=80, kde=True, color="#4C78A8", ax=axes[1])
axes[1].set_title("Histogram of future_return (before split)")
axes[1].set_xlabel("future_return")

plt.tight_layout()
plt.show()

imbalance_ratio = class_counts.max() / class_counts.min()
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
print("Split recommendation: use chronological split for time series (not random stratified split).")
if imbalance_ratio >= 2.0:
    print("WeightedRandomSampler recommendation: YES (strong imbalance).")
elif imbalance_ratio >= 1.3:
    print("WeightedRandomSampler recommendation: OPTIONAL (run ablation with and without weights).")
else:
    print("WeightedRandomSampler recommendation: NOT REQUIRED (classes relatively balanced).")


### 7. Build features and split datasets


In [ ]:
def prepare_datasets(
    data_config: DataConfig,
    target_config: TargetConfig,
    split_config: SplitConfig,
) -> dict:
    raw_df_local = pd.read_csv(data_config.raw_csv)
    clean_df_local, cleaning_stats_local = _clean_ohlcv(raw_df_local)
    _save_dataframe(clean_df_local, data_config.clean_csv)

    feat_df = _create_basic_features(clean_df_local)
    feat_df = _make_multiclass_target(
        feat_df,
        horizon=target_config.horizon,
        threshold=target_config.threshold,
    )
    feat_df = feat_df.dropna().copy()

    target_series = feat_df["target"].astype(int)
    future_return = feat_df["future_return"].copy()
    features_only = feat_df.drop(columns=["target", "future_return"])
    features_only = features_only.select_dtypes(include=["number"]).copy()

    final_df = features_only.copy()
    final_df["future_return"] = future_return.loc[final_df.index]
    final_df["target"] = target_series.loc[final_df.index]
    final_df = final_df.dropna().copy()

    train_df, val_df, test_df = _chronological_split(
        final_df,
        train_ratio=split_config.train_ratio,
        val_ratio=split_config.val_ratio,
        test_ratio=split_config.test_ratio,
    )

    _save_dataframe(final_df, data_config.features_parquet)
    _save_dataframe(train_df, data_config.train_parquet)
    _save_dataframe(val_df, data_config.val_parquet)
    _save_dataframe(test_df, data_config.test_parquet)

    class_counts_local = final_df["target"].value_counts().sort_index().to_dict()
    metadata = {
        "cleaning_stats": cleaning_stats_local,
        "rows_final_dataset": int(len(final_df)),
        "split_sizes": {
            "train": int(len(train_df)),
            "val": int(len(val_df)),
            "test": int(len(test_df)),
        },
        "class_counts": {str(k): int(v) for k, v in class_counts_local.items()},
        "num_features": int(len(features_only.columns)),
        "feature_columns": list(features_only.columns),
        "target_config": {
            "horizon": int(target_config.horizon),
            "threshold": float(target_config.threshold),
        },
        "notebook_mode": "simple_local_pipeline",
    }

    reports_dir = PROJECT_ROOT / "reports"
    reports_dir.mkdir(parents=True, exist_ok=True)
    with (reports_dir / "data_prep_metadata.json").open("w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    return metadata


In [ ]:
data_cfg = DataConfig()
target_cfg = TargetConfig(horizon=6, threshold=0.0075)
split_cfg = SplitConfig(train_ratio=0.70, val_ratio=0.15, test_ratio=0.15)

metadata = prepare_datasets(
    data_config=data_cfg,
    target_config=target_cfg,
    split_config=split_cfg,
)

display(pd.Series(metadata, name="metadata"))


### 8. Post-split sanity checks


In [ ]:
cfg = DataConfig()
train_df = pd.read_parquet(cfg.train_parquet)
val_df = pd.read_parquet(cfg.val_parquet)
test_df = pd.read_parquet(cfg.test_parquet)

print("train/val/test rows:", len(train_df), len(val_df), len(test_df))

split_class_dist = pd.DataFrame(
    {
        "train": train_df["target"].value_counts().sort_index(),
        "val": val_df["target"].value_counts().sort_index(),
        "test": test_df["target"].value_counts().sort_index(),
    }
).fillna(0).astype(int)

split_class_dist.index = split_class_dist.index.map(lambda x: label_map.get(x, str(x)))
display(split_class_dist)

split_class_pct = split_class_dist.div(split_class_dist.sum(axis=0), axis=1) * 100
split_class_pct.T.plot(kind="bar", stacked=True, figsize=(8, 4), colormap="Blues")
plt.title("Class share by split")
plt.xlabel("split")
plt.ylabel("share [%]")
plt.legend(title="class", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()
